# 02 — Continental aerial rivers: six precipitationsheds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/blob/main/notebooks/02_aerial_rivers_precipitationsheds.ipynb)

**What we're investigating.** Evapotranspiration from the Amazon basin
sustains rainfall in remote downwind regions through "aerial rivers" of
atmospheric moisture. Because this dataset tags *every* 0.25° land grid
cell as an independent receptor, any single grid cell can be treated as
a localised sink and its full 30-year climatological **precipitationshed**
reconstructed — no pre-defined river basins required.

This notebook contrasts the **1990–2019 mean precipitationsheds** of six
receptor cells spanning distinct climate regimes:

- **Manaus**, **Santa Cruz de la Sierra**, and **São Paulo** — Amazon-fed
  sinks that expose the continental aerial rivers of Amazonian
  evapotranspiration sustaining downwind rainfall.
- **Santiago** — a Pacific-facing sink drawing almost entirely on oceanic,
  westerly-transported moisture.
- **Buenos Aires** and **Viedma** — eastern-slope sinks fed by the
  continental La Plata / South American Low-Level Jet corridor, with Viedma
  receiving a more mixed continental, Atlantic, and trans-Andean Pacific
  supply.

This reproduces **Figure 6** of the manuscript.

**What you'll get.** A single six-panel (2×3) figure,
`multi_precipitationsheds.png`. Each panel is a **per-panel spatial
cumulative distribution** of that cell's 30-year mean tracked-evaporation
source field, ranked so that 0 % marks the single strongest source cell and
100 % the diffuse tail — the dark core traces the region supplying most of
each location's rainfall. A red star marks each receptor cell.

**Dataset.** AguaTrack, **yearly aggregate** zarr store
(`AguaTrack_ARCO_SA_yearly.zarr`). We open the store once and, following the
access rules, select the six tag cells along the fast `tagging_mask` axis
*before* materialising anything — a tiny memory footprint.

**How to cite.** See the [repo README](https://github.com/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial#how-to-cite).

## Step 1 — Configuration

Everything you might want to edit lives in this single cell:

- **HuggingFace dataset** — the consolidated yearly zarr that holds all
  30 years on one `time` axis.
- **Receptor cells** — the six `(name, lat, lon)` targets. Add, remove, or
  move any of them; the analysis re-tags automatically. (The 2×3 layout
  below assumes six cells.)
- **Map extent** — the union of the six cities' source footprints, clamped
  to the data domain.

In [ ]:
HF_REVISION = "main"

# Yearly aggregate store on HuggingFace.
AGUATRACK_YEARLY_URL = (
    "hf://datasets/AguaTrackSA/AguaTrack-ARCO-SA-Aggregated"
    "/AguaTrack_ARCO_SA_yearly.zarr"
)
# For a LOCAL run, point this at the on-disk mirror and drop
# `storage_options` in Step 4, e.g.:
#   AGUATRACK_YEARLY_URL = "/path/to/AguaTrack_ARCO_SA_yearly.zarr"

# Six climatically contrasting receptor cells (name, lat, lon).
CITIES = [
    ("Manaus", -3.10, -60.02),
    ("Santa Cruz", -17.80, -63.20),
    ("Sao Paulo", -23.55, -46.63),
    ("Santiago", -33.45, -70.67),
    ("Buenos Aires", -34.60, -58.40),
    ("Viedma", -40.82, -63.00),
]

# Map extent [lon_min, lon_max, lat_min, lat_max] — union of the six cities'
# source footprints, clamped to the data domain (lon >= -90).
MAP_EXTENT = [-90, -25, -51, 10]

## Step 2 — Install dependencies (Colab only)

Colab gets a fresh runtime every session, so we install the geo stack.
Local users skip this — `uv sync` already provided the deps.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        'install -q cartopy cmcrameri "xarray>=2026" "zarr>=3" '
        "fsspec huggingface_hub dask",
    )

## Step 3 — Imports and plotting style

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import xarray as xr

plt.rcParams.update({
    "font.size": 18, "axes.titlesize": 18, "axes.labelsize": 18,
    "xtick.labelsize": 15, "ytick.labelsize": 15, "legend.fontsize": 15,
})

## Step 4 — Find each receptor's nearest tag and load its 30-year mean

`tag_lat` / `tag_lon` are static metadata of the tracking domain. A simple
Euclidean argmin in (lat, lon) gives the nearest tagged grid cell — at
0.25° resolution this is precise enough; no great-circle distance needed.

Crucially, we select all six tags along the fast `tagging_mask` axis
**before** calling `.load()`, then average over `time` (the yearly store's
30 annual steps) to get each cell's 30-year mean source field. This reads
only the six required chunks.

In [ ]:
ds = xr.open_zarr(AGUATRACK_YEARLY_URL,
                  storage_options={"revision": HF_REVISION})

idxs, tag_ll = [], []
for name, lat, lon in CITIES:
    i = int(np.asarray(((ds.tag_lat - lat) ** 2 + (ds.tag_lon - lon) ** 2).argmin()))
    idxs.append(i)
    tag_ll.append((float(ds.tag_lat.isel(tagging_mask=i)),
                   float(ds.tag_lon.isel(tagging_mask=i))))
    print(f"  {name:26s} -> tag ({tag_ll[-1][0]:.2f}, {tag_ll[-1][1]:.2f})  idx={i}")

# 30-year mean source field for the six tags at once: (6, lat, lon).
clim = ds.e_track.isel(tagging_mask=idxs).mean("time").load()
ds.close()
print(f"loaded {clim.sizes['tagging_mask']} source maps "
      f"({clim.nbytes / 1e6:.1f} MB in RAM)")

## Step 5 — Per-panel spatial cumulative distribution

For each cell we rank all source pixels (land + ocean) from strongest to
weakest and take the running share of the total tracked evaporation. The
result is a 0–100 % map where 0 % is the single strongest source cell and
100 % the diffuse tail — the same encoding used for the monthly
precipitationsheds in this dataset's other figures.

In [ ]:
def spatial_cdf(da: xr.DataArray) -> xr.DataArray:
    flat = da.values.flatten()
    valid = ~np.isnan(flat) & (flat > 0)
    vals = flat[valid]
    order = np.argsort(vals)[::-1]
    cum = np.minimum(np.cumsum(vals[order]) / np.sum(vals[order]) * 100, 100.0)  # cap at 100%
    out = np.full_like(flat, np.nan, dtype=float)
    out[np.where(valid)[0][order]] = cum
    return xr.DataArray(out.reshape(da.values.shape), coords=da.coords, dims=da.dims)

## Step 6 — Six-panel precipitationshed figure

Top row: Manaus, Santa Cruz, São Paulo. Bottom row: Santiago, Buenos Aires,
Viedma. A shared horizontal colour bar reports the cumulative moisture
contribution (%); the red star marks each receptor cell.

In [ ]:
levels = np.arange(0, 101, 10)


def style_ax(ax, left_labels, bottom_labels):
    ax.coastlines(resolution="50m", color="black", linewidth=0.7)
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.0)
    gl.top_labels = gl.right_labels = False
    gl.left_labels = left_labels
    gl.bottom_labels = bottom_labels
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 20))
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, 20))
    gl.xlabel_style = gl.ylabel_style = {"size": 15}


fig, axes = plt.subplots(2, 3, figsize=(12, 9.6),
                         subplot_kw={"projection": ccrs.PlateCarree()})
axes = axes.flatten()
cf = None
for i, ((name, _, _), (tlat, tlon)) in enumerate(zip(CITIES, tag_ll)):
    ax = axes[i]
    cf = spatial_cdf(clim.isel(tagging_mask=i)).plot.contourf(
        ax=ax, levels=levels, cmap=cmc.batlowW,
        transform=ccrs.PlateCarree(), add_colorbar=False, extend="neither")
    ax.scatter(tlon, tlat, color="red", s=160, marker="*", edgecolor="white",
               linewidth=0.8, transform=ccrs.PlateCarree(), zorder=6)
    ax.set_title(f"{name}\n({tlon:.2f}, {tlat:.2f})", fontsize=24)
    style_ax(ax, left_labels=(i % 3 == 0), bottom_labels=(i >= 3))

fig.subplots_adjust(left=0.05, right=0.97, top=0.94, bottom=0.10,
                    hspace=0.16, wspace=0.06)
cbar_ax = fig.add_axes([0.25, 0.055, 0.5, 0.015])
cb = fig.colorbar(cf, cax=cbar_ax, orientation="horizontal", ticks=levels,
                  extend="neither",
                  label="Cumulative Moisture Contribution (%)")
cb.ax.tick_params(labelsize=15)

OUT = Path("outputs/aerial_river"); OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "multi_precipitationsheds.png", bbox_inches="tight", dpi=200)
plt.show()